# Privacy, Governance & Compliance — NovaCred Credit Applications

## Purpose
This notebook examines the NovaCred dataset through a data governance and regulatory compliance lens. It identifies all PII fields, demonstrates pseudonymization, maps findings to GDPR provisions, references the EU AI Act, and proposes concrete governance controls.

## Dataset
- **Source:** `../data/cleaned_credit_applications.csv`
- **Context:** Credit application records from NovaCred fintech platform

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re

df = pd.read_csv('../data/cleaned_credit_applications.csv')

print(f'Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns')
print(f'\nAll columns:')
for col in df.columns:
    print(f'  - {col}')

---

## 2. PII Field Identification

Personally Identifiable Information (PII) refers to any data that can be used to identify a specific individual, either directly or in combination with other fields.

We classify PII into three categories:
- **Direct identifiers**: uniquely identify a person on their own (e.g. name, SSN)
- **Quasi-identifiers**: identify a person when combined with other fields (e.g. zip code + DOB)
- **Sensitive behavioral data**: reveals patterns that can infer protected characteristics

In [ ]:
pii_fields = {
    'applicant_info.full_name':     ('Direct identifier',         'High'),
    'applicant_info.email':         ('Direct identifier',         'High'),
    'applicant_info.ssn':           ('Sensitive identifier',      'Critical'),
    'applicant_info.ip_address':    ('Quasi-identifier',          'Medium'),
    'applicant_info.date_of_birth': ('Quasi-identifier',          'Medium'),
    'applicant_info.zip_code':      ('Quasi-identifier',          'Low'),
    'spending_behavior':            ('Sensitive behavioral data', 'Medium-High'),
}

print('PII Field Identification')
print('=' * 80)
print(f'{"Field":<40} {"PII Type":<30} {"Risk":<12} {"Present in Dataset"}')
print('-' * 80)

for field, (pii_type, risk) in pii_fields.items():
    present = field in df.columns
    non_null = df[field].notna().sum() if present else 'N/A'
    status = f'{non_null} records' if present else 'NOT FOUND'
    print(f'{field:<40} {pii_type:<30} {risk:<12} {status}')

print(f'\nTotal PII fields identified: {len(pii_fields)}')

### Note on `spending_behavior`

`spending_behavior` is flagged as **sensitive behavioral data**. While not a direct identifier, it reveals patterns about a person's financial habits and lifestyle, which can be used to infer protected characteristics (e.g. religion, health status). Under GDPR, behavioral data that carries this inferential risk should be treated with the same caution as traditional PII.